# SolarGuard — Explore the X6.3 Flare (Feb 22, 2024)
Run this notebook to visually understand your data before running the pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

ROOT = Path('..') 
print('ROOT:', ROOT.resolve())

In [ ]:
# Load live GOES data (run step1 first)
soft = pd.read_csv(ROOT / 'data/raw/goes/live_soft_xray.csv', parse_dates=['time_tag'])
hard = pd.read_csv(ROOT / 'data/raw/goes/live_hard_xray.csv', parse_dates=['time_tag'])
print(f'Soft: {len(soft)} rows | Hard: {len(hard)} rows')
soft.head()

In [ ]:
# Plot dual light curves
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('SolarGuard — GOES XRS Light Curves', fontsize=14, fontweight='bold')

ax1.semilogy(soft['time_tag'], soft['flux'], color='#F4A300', linewidth=1)
ax1.set_ylabel('Soft X-ray Flux\n(W/m²)')
ax1.set_title('SoLEXS proxy (GOES soft X-ray 0.05–0.4 nm)')
ax1.grid(alpha=0.3)

ax2.semilogy(hard['time_tag'], hard['flux'], color='#D04B20', linewidth=1)
ax2.set_ylabel('Hard X-ray Flux\n(W/m²)')
ax2.set_title('HEL1OS proxy (GOES hard X-ray 0.1–0.8 nm)')
ax2.grid(alpha=0.3)

# Spectral hardening ratio
merged = pd.merge_asof(soft.sort_values('time_tag'), hard.sort_values('time_tag'),
                        on='time_tag', suffixes=('_soft','_hard'))
ratio = merged['flux_hard'] / (merged['flux_soft'] + 1e-30)
ax3.plot(merged['time_tag'], ratio, color='#1D9E75', linewidth=1)
ax3.set_ylabel('Spectral\nHardening Ratio')
ax3.set_title('Hard/Soft Ratio — THE KEY PRECURSOR SIGNAL')
ax3.grid(alpha=0.3)

ax3.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(ROOT / 'data/processed/flare_visualization.png', dpi=150)
plt.show()
print('Plot saved to data/processed/flare_visualization.png')

In [ ]:
# Compute spectral hardening ratio features
import sys
sys.path.insert(0, str(ROOT))
from pipeline.step3_feature_engine import compute_spectral_hardening_ratio, compute_rolling_stats

df = merged.rename(columns={'flux_soft': 'soft_flux', 'flux_hard': 'hard_flux'})
df = compute_spectral_hardening_ratio(df)
df = compute_rolling_stats(df)

print('Features computed:')
print(df[['soft_flux','hard_flux','spectral_hardening_ratio','spectral_hardening_ratio_zscore']].describe())